<a href="https://colab.research.google.com/github/bsrikanth24/Best-websites-a-programmer-should-visit/blob/master/Missing_sequence_numbers_in_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import min, max, col

spark = SparkSession.builder.appName("FindMissingNumbers").getOrCreate()
df = spark.createDataFrame([(x,) for x in [1, 2, 4, 5, 9, 10]], ["num"])
df.show()

# Find min and max
# min_num = df.agg({"num": "min"}).first()[0]
# max_num = df.agg({"num": "max"}).first()[0]

# Import the functions (you already did this)
# from pyspark.sql.functions import min, max

# Get both in one scan, extract them cleanly
min_max_row = df.select(min("num"), max("num")).first()
min_num = min_max_row[0]
max_num = min_max_row[1]
# min_max_row.show(truncate=False) # This line caused the error


# Generate full sequence DataFrame
full_seq = spark.range(min_num, max_num + 1).toDF("num")
full_seq.show(truncate=False)

# Left anti join to find missing numbers
missing = full_seq.join(df, on="num", how="left_anti")
missing.show()

# How .first()[0] works:
# df.agg(...) returns a new DataFrame with one row and one column containing the result.
# .first() extracts that single row as a PySpark Row object (e.g., Row(min(num)=1)).
# [0] extracts the actual integer value (1) from that Row object so we can use it in standard Python math later.

# What it does: Creates a brand new DataFrame containing every single integer from the minimum to the maximum.
# spark.range(start, end): This is a highly optimized Spark function that generates a sequence of numbers.
# Why max_num + 1? The range function is exclusive at the end (just like Python's native range()). If we want 10 to be included, we must pass 11 as the end parameter.
# .toDF("num"): spark.range creates a column named id by default. This renames it to num so it matches the original DataFrame.

+---+
|num|
+---+
|  1|
|  2|
|  4|
|  5|
|  9|
| 10|
+---+

+---+
|num|
+---+
|1  |
|2  |
|3  |
|4  |
|5  |
|6  |
|7  |
|8  |
|9  |
|10 |
+---+

+---+
|num|
+---+
|  3|
|  7|
|  6|
|  8|
+---+

